[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-hierarchical.ipynb)

# Hierarchical Clustering & DBSCAN

*AIBits Academy · Machine Learning End To End · Unsupervised Learning · New*

Two clustering paradigms that free you from K-Means' biggest constraints — knowing k in advance, and assuming round, evenly-sized clusters.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Agglomerative Hierarchical Clustering

Start with every point as its own cluster. Repeatedly merge the two *closest* clusters until only one remains, recording every merge. The result is a full nested hierarchy — you choose how many clusters you want *after* seeing the structure, by cutting the tree at any height.

- Compute pairwise distances between all n points → n×n distance matrix

- Merge the two closest clusters into one

- Update distances from the new cluster to all others, using a **linkage criterion**

- Repeat steps 2–3 until one cluster remains — record the merge order and heights

## Linkage Criteria — How to Measure Distance Between Clusters

| Linkage | Formula (A, B = two clusters) | Tendency |
|---|---|---|
| **Single** | min <sub>a∈A, b∈B</sub> d(a,b) | Chains together — can form long, straggly clusters ("chaining effect") |
| **Complete** | max <sub>a∈A, b∈B</sub> d(a,b) | Compact, roughly equal-sized clusters; sensitive to outliers |
| **Average** | (1/\|A\|\|B\|) Σₐ Σᵦ d(a,b) | Balanced compromise between single and complete |
| **Ward** | Minimises increase in total within-cluster variance (SSE) after merge | Default choice — produces compact, roughly K-Means-like clusters |

## Dendrogram — Reading the Merge Tree

Cutting the dendrogram at a given height y is equivalent to asking: "stop merging once the closest two remaining clusters are farther apart than y." Cut low → many small clusters; cut high → few large clusters. Crucially, you make this choice *after* inspecting the whole tree, unlike K-Means where k must be fixed up front.

## Watch the Dendrogram Build — Points Merge, Tree Grows

Agglomerative clustering in motion (Ward linkage, verified against SciPy) on 8 Mumbai localities. The two panels are synchronised: on the left, the two closest clusters merge and take a shared colour; on the right, the corresponding bracket is drawn at a height equal to the merge distance. Watch the nearby pairs join first at low height, then whole regions combine at greater heights — the tall final links are exactly where you'd cut to choose k.

## Code — Agglomerative Clustering, Mumbai Locality Data

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

# Mumbai localities: [avg_rent_k, commute_min, greenery_pct]
np.random.seed(3)
localities = ['Bandra','Andheri','Powai','Thane','Panvel','Kalyan','Worli','Borivali']
X = np.array([
    [85,35,12],[70,40,15],[55,45,30],[30,65,35],
    [22,75,40],[25,70,32],[95,30,8],[45,50,38]
])
Xs = StandardScaler().fit_transform(X)

Z = linkage(Xs, method='ward')
labels3 = fcluster(Z, t=3, criterion='maxclust')
for name, lab in zip(localities, labels3):
    print(f"  {name:10s} → cluster {lab}")

# Equivalent via sklearn
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
print("\nsklearn labels:", agg.fit_predict(Xs))

## DBSCAN — Density-Based Clustering

DBSCAN groups points that are densely packed together and explicitly labels sparse, isolated points as **noise** — a capability neither K-Means nor hierarchical clustering has natively (every point in those methods is forced into some cluster).

**Two parameters define density:** 
`ε (eps)` — neighbourhood radius 
`minPts` — minimum points required within ε to call a region "dense" 
 
**Three point types:** 
**Core point** — has ≥ minPts neighbours within ε 
**Border point** — within ε of a core point, but doesn't itself have minPts neighbours 
**Noise point** — neither core nor border — an outlier

- Pick an unvisited point. If it's a core point, start a new cluster

- Add all points density-reachable from it (recursively expand through neighbouring core points) to the cluster

- Points that end up reachable from no core point are labelled noise (−1)

- Repeat until every point is visited

## Watch DBSCAN Grow — Core, Border & Noise

This runs a real DBSCAN (ε and minPts verified against scikit-learn) on 48 points forming one arc-shaped cluster, one dense blob, and scattered noise. Press Play and watch clusters *grow outward* from core points: each core point (filled) pulls in every neighbour within its ε-radius (yellow halo); border points (rings) join a cluster but don't expand it further; and points that never fall within ε of any core point are left over as noise (red). This density-reachability growth is exactly why DBSCAN handles the non-spherical arc that K-Means would tear apart.

## Code — DBSCAN for Outlier-Aware Customer Segmentation

Surat textile B2B customers — most cluster into regular ordering patterns, but a few place wildly irregular bulk orders that should be flagged, not force-fit into a cluster:

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import numpy as np

np.random.seed(11)
# Regular customers: 3 tight clusters. Plus 8 erratic bulk-order outliers.
c1 = np.random.normal([20,5], 2, (60,2))    # small frequent orders
c2 = np.random.normal([60,15], 3, (50,2))   # medium orders
c3 = np.random.normal([100,8], 2.5, (40,2))  # bulk, low frequency
outliers = np.random.uniform([0,0], [130,30], (8,2))
X = np.vstack([c1,c2,c3,outliers])
Xs = StandardScaler().fit_transform(X)

# Choose eps via k-distance elbow (k = minPts)
minPts = 5
nn = NearestNeighbors(n_neighbors=minPts).fit(Xs)
dists,_ = nn.kneighbors(Xs)
k_dist = np.sort(dists[:,-1])
# Elbow typically around the 90th percentile of sorted k-distances
eps_guess = np.percentile(k_dist, 90)
print(f"Suggested eps ≈ {eps_guess:.3f}")

db = DBSCAN(eps=eps_guess, min_samples=minPts).fit(Xs)
n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_noise = list(db.labels_).count(-1)
print(f"Clusters found: {n_clusters}   Noise points flagged: {n_noise}")

DBSCAN correctly recovers the 3 genuine customer segments *and* separately flags 7 of the 8 erratic bulk-order accounts as noise — exactly the outliers a business analyst would want surfaced for manual review, not silently averaged into a cluster centroid.

## Choosing ε — The K-Distance Elbow Plot

Plot the distance to each point's k-th nearest neighbour (k = minPts), sorted ascending. Points in dense regions have small k-distances; points near sparse/noise regions have large ones. The "elbow" — where the curve sharply bends upward — is a good ε: below it, points are considered core; above it, they're too isolated.

## Choosing the Right Clustering Algorithm

|  | K-Means | Hierarchical | DBSCAN |
|---|---|---|---|
| Need k upfront? | Yes | No — choose after seeing dendrogram | No — discovers cluster count automatically |
| Cluster shape assumption | Convex, roughly spherical | Depends on linkage (Ward ≈ spherical) | Arbitrary shape, handles non-convex clusters |
| Handles outliers? | No — every point forced into a cluster | No | Yes — explicit noise label |
| Scalability | O(nkdi) — scales well | O(n²) or O(n³) — slow beyond ~10k points | O(n log n) with spatial index |
| Deterministic? | No (depends on init, unless K-Means++) | Yes | Yes |

> **🔗 Real-World Link — Customer Segmentation**
>
> 8,950 credit card holders, clustered on 18 real usage features. Silhouette score (not the elbow curve) is what correctly picks k=3 here, uncovering a "cash-advance-reliant" segment worth treating very differently from ordinary big spenders. [See the case study →](https://statso.io/2022/11/23/customer-segmentation-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Cut a dendrogram

Build the `linkage(X, method="ward")` and cut it into 2 flat clusters with `fcluster(..., t=2, criterion="maxclust")`. Store the sorted cluster sizes in `sizes`.

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
X = np.array([[1, 1], [1.2, 0.8], [0.9, 1.1], [6, 6], [6.2, 5.9], [5.8, 6.1]])
sizes = None   # TODO


In [ ]:
try:
    check("two groups of three", sizes == [3, 3])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
X = np.array([[1, 1], [1.2, 0.8], [0.9, 1.1], [6, 6], [6.2, 5.9], [5.8, 6.1]])
Z = linkage(X, method="ward")
labels = fcluster(Z, t=2, criterion="maxclust")
sizes = sorted(np.bincount(labels)[1:].tolist())

```

</details>

### Exercise 2 · Medium · Which linkage represents the data best?

The cophenetic correlation says how faithfully a dendrogram preserves the original distances. Compute it for `single`, `complete`, `average` and `ward` linkage on `X2` (`scipy.cluster.hierarchy.cophenet` with `pdist`), store the dict in `coph`, and the best method name in `best_method`.

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, cophenet
from scipy.spatial.distance import pdist
rng = np.random.default_rng(0)
X2 = np.vstack([rng.normal(0, 1, (30, 2)), rng.normal(6, 1, (30, 2))])
coph = {}
best_method = None   # TODO


In [ ]:
try:
    check("four methods", set(coph) == {"single", "complete", "average", "ward"})
    check("all are decent (> 0.6)", min(coph.values()) > 0.6)
    check("best is the max", best_method == max(coph, key=coph.get))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from scipy.cluster.hierarchy import linkage, cophenet
from scipy.spatial.distance import pdist
rng = np.random.default_rng(0)
X2 = np.vstack([rng.normal(0, 1, (30, 2)), rng.normal(6, 1, (30, 2))])
coph = {m: cophenet(linkage(X2, method=m), pdist(X2))[0] for m in ["single", "complete", "average", "ward"]}
best_method = max(coph, key=coph.get)

```

</details>

### Exercise 3 · Stretch · DBSCAN finds clusters and noise

Run `DBSCAN(eps=0.8, min_samples=5)` on `X3`. Store the number of clusters (ignoring noise label -1) in `n_clusters` and the number of noise points in `n_noise`.

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs
X3, _ = make_blobs(n_samples=200, centers=[[0, 0], [7, 7]], cluster_std=0.4, random_state=0)
X3 = np.vstack([X3, [[3.5, 3.5], [-4, 6]]])
n_clusters = n_noise = None   # TODO


In [ ]:
try:
    check("two dense clusters", n_clusters == 2)
    check("the two strays are noise", n_noise == 2)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs
X3, _ = make_blobs(n_samples=200, centers=[[0, 0], [7, 7]], cluster_std=0.4, random_state=0)
X3 = np.vstack([X3, [[3.5, 3.5], [-4, 6]]])
labels = DBSCAN(eps=0.8, min_samples=5).fit_predict(X3)
n_clusters = len(set(labels) - {-1})
n_noise = int((labels == -1).sum())

```

Unlike k-means, DBSCAN needs no k, handles odd shapes, and labels outliers as noise.

</details>

---
*Back to the course: **Machine Learning End To End → Hierarchical Clustering & DBSCAN**.*